# Download Spain

In [3]:
import requests
import xml.etree.ElementTree as ET
import zipfile
import io
from io import BytesIO
import geopandas as gpd
import pandas as pd
import os
import time
import fiona
import re
import json
import requests

from pathlib import Path

In [10]:
BASE_URL = "https://www.catastro.hacienda.gob.es/INSPIRE/buildings/ES.SDGC.BU.atom.xml"
OUT_DIR_MAIN = "/data/uscuni-ulce/extension/spain"
CAT_DOMAIN = "https://www.catastro.hacienda.gob.es/INSPIRE/"

In [ ]:
def get_feed_entries(url):
    """Return list of dicts with 'title', 'link', 'id' from an ATOM feed."""
    try:
        r = requests.get(url, timeout=30)  # ⬅ timeout avoids infinite hanging
        r.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Skipping feed {url} due to error: {e}")
        return []

    root = ET.fromstring(r.content)
    ns = {"atom": "http://www.w3.org/2005/Atom"}

    entries = []
    for entry in root.findall("atom:entry", ns):
        title = entry.find("atom:title", ns).text
        link = entry.find("atom:link", ns).attrib["href"]
        id_elt = entry.find("atom:id", ns)
        entries.append({
            "title": title,
            "link": link,
            "id": id_elt.text if id_elt is not None else None,
        })
    return entries

provinces = get_feed_entries(BASE_URL)
print(provinces)

In [ ]:
for p_idx, prov in enumerate(provinces):

    print(f"\nProvince {p_idx}: {prov['title']}")

    # Step 2: get municipalities in this province
    municipalities = get_feed_entries(prov["link"])
    print(f"  -> {len(municipalities)} municipalities")

    for m_idx, muni in enumerate(municipalities):
        # Generate a safe filename for each municipality
        muni_id = muni["id"] if muni["id"] else f"muni_{p_idx:02d}_{m_idx:04d}"
        muni_id_safe = muni_id.replace("/", "_")
        
        try:
            r = requests.get(muni["link"], timeout=120)
            r.raise_for_status()
        except Exception as e:
            print(f"    Download failed: {e}")
            continue

        try:
            with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                building_files = [m for m in z.namelist() if m.lower().endswith("building.gml")]
                for member in building_files:
                    # Save using just the GML filename
                    out_path = os.path.join(OUT_DIR_MAIN, os.path.basename(member))
                    with z.open(member) as src, open(out_path, "wb") as dst:
                        dst.write(src.read())
                    print(f"Saved {out_path}")

        except Exception as e:
            print(f"    Failed processing {muni_id}: {e}")

        time.sleep(0.2)  # polite delay


## Download Basque Country separately

Basque Country provinces had to be downloaded separately.
- Araba/Alava: https://geo.araba.eus/deskargak/INSPIRE/BU/GML/3042/BU_3042_GML.zip
- Gipuzkoa: https://b5m.gipuzkoa.eus/inspire/download/GML/ES.GFA.BU.zip
- Bizkaia: code below

In [11]:
OUT_DIR_BASQUE = "/data/uscuni-ulce/extension/basque"

### Bizkaia

In [15]:
# Loop from 1 to 915
for idx in range(1,915):
    # Make URL with leading zeros (e.g., 001, 002)
    zip_url = f"https://apli.bizkaia.eus/apps/Danok/INSPIRE/ES.BFA.BU.{str(idx).zfill(3)}.zip"
    print(f"Downloading {zip_url} ...")
    
    try:
        r = requests.get(zip_url, timeout=60)
        r.raise_for_status()
    except Exception as e:
        print(f"  -> Failed to download {zip_url}: {e}")
        continue

    # Open ZIP in memory
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        # Find GML file
        gml_file_name = [f for f in z.namelist() if f.lower().endswith(".gml")]
        if not gml_file_name:
            print(f"  -> No GML found in {zip_url}")
            continue
        
        gml_file_name = gml_file_name[0]
        
        # Extract GML and save with unique name
        new_gml_name = f"ES.BFA.BU.{str(idx).zfill(3)}.gml"
        gml_path = os.path.join(OUT_DIR_BASQUE, new_gml_name)
        
        with z.open(gml_file_name) as gml_file, open(gml_path, "wb") as f_out:
            f_out.write(gml_file.read())
        
        print(f"  -> Saved {gml_path}")

  -> Saved /data/uscuni-ulce/extension/basque/ES.BFA.BU.020.gml


Navarra

In [8]:
WFS = "https://inspire.navarra.es/services/BU/wfs"
CAP = 5000            # server's max features per request
MIN_SIZE = 50         # metres; stop splitting below this
OUT = "/data/uscuni-ulce/extension/basque/navarra/navarra_buildings.gpkg"
CRS = "EPSG:25830"

# Extent of Navarra in EPSG:25830 (xmin, ymin, xmax, ymax)
EXTENT = (541000, 4638000, 679000, 4800000)

features = {}         # feature id -> feature (dedupes buildings crossing tile edges)
capped = []           # tiles that still hit the cap at minimum size


def fetch(bbox, tries=5):
    xmin, ymin, xmax, ymax = bbox
    params = dict(
        service="WFS", version="2.0.0", request="GetFeature",
        typeNames="BU:Building",
        srsName=CRS,
        outputFormat="application/json",
        count=CAP,
        bbox=f"{xmin},{ymin},{xmax},{ymax},urn:ogc:def:crs:EPSG::25830",
    )
    for i in range(tries):
        try:
            r = requests.get(WFS, params=params, timeout=300)
            r.raise_for_status()
            return r.json()["features"]
        except Exception as e:
            print(f"  retry {i + 1} for {bbox}: {e}")
            time.sleep(5 * (i + 1))
    raise RuntimeError(f"Failed: {bbox}")


def download(bbox):
    xmin, ymin, xmax, ymax = bbox
    feats = fetch(bbox)
    if not feats:
        return
    if len(feats) >= CAP and (xmax - xmin) > MIN_SIZE:
        # tile is capped: split in 4 and recurse
        xm, ym = (xmin + xmax) / 2, (ymin + ymax) / 2
        for b in [(xmin, ymin, xm, ym), (xm, ymin, xmax, ym),
                  (xmin, ym, xm, ymax), (xm, ym, xmax, ymax)]:
            download(b)
        return
    if len(feats) >= CAP:
        capped.append(bbox)
    for f in feats:
        features[f["id"]] = f
    print(f"tile {bbox}: {len(feats)} (total unique: {len(features)})")


# Start with 20 km tiles over the whole extent
STEP = 20000
x = EXTENT[0]
while x < EXTENT[2]:
    y = EXTENT[1]
    while y < EXTENT[3]:
        download((x, y, x + STEP, y + STEP))
        y += STEP
    x += STEP

print("Unique buildings:", len(features))
if capped:
    print("WARNING: tiles still capped at minimum size:", capped)

# Build GeoDataFrame, keeping the feature id
rows = list(features.values())
for f in rows:
    f["properties"]["feature_id"] = f["id"]
gdf = gpd.GeoDataFrame.from_features(rows, crs=CRS)

# INSPIRE attributes can be nested; GeoPackage needs plain values
for col in gdf.columns:
    if col != "geometry" and gdf[col].map(lambda v: isinstance(v, (dict, list))).any():
        gdf[col] = gdf[col].map(lambda v: json.dumps(v, ensure_ascii=False)
                                if isinstance(v, (dict, list)) else v)

gdf.to_file(OUT, driver="GPKG", layer="buildings")
print(f"Saved {len(gdf)} buildings to {OUT}")

tile (541000, 4698000, 561000, 4718000): 3698 (total unique: 3698)
tile (541000, 4718000, 561000, 4738000): 1521 (total unique: 5168)
tile (561000, 4678000, 581000, 4698000): 3089 (total unique: 8257)
tile (561000, 4698000, 571000.0, 4708000.0): 2436 (total unique: 10693)
tile (571000.0, 4698000, 581000, 4708000.0): 1044 (total unique: 11737)
tile (561000, 4708000.0, 571000.0, 4718000): 1132 (total unique: 12869)
tile (571000.0, 4708000.0, 581000, 4718000): 2566 (total unique: 15435)
tile (561000, 4718000, 571000.0, 4728000.0): 1794 (total unique: 17227)
tile (571000.0, 4718000, 581000, 4728000.0): 4970 (total unique: 22189)
tile (561000, 4728000.0, 571000.0, 4738000): 892 (total unique: 23079)
tile (571000.0, 4728000.0, 581000, 4738000): 1486 (total unique: 24553)
tile (561000, 4738000, 571000.0, 4748000.0): 796 (total unique: 25349)
tile (571000.0, 4738000, 581000, 4748000.0): 362 (total unique: 25711)
tile (561000, 4748000.0, 571000.0, 4758000): 1823 (total unique: 27532)
tile (5710